In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow.feather as feather
import seaborn as sns
import seaborn.objects as so
from scipy.stats import kstest

import importlib
import sys
import os
hostname = os.uname().nodename
if hostname == 'BlackBeast':
    path = '/home/hedvigs/snake_book/econ'
    site = 'home'
elif hostname == 'hedvig-hp-elitedesk-800-g5-twr':
    path = '/home/hedvigs/PycharmProjects/homewrs/plab_workflow/workflow/'
    site = 'work'
elif hostname == 'work-computer':
    path = '/mnt/work/workbench/hedvigs/snake_book/econ'
    site = 'server'
elif hostname == 'wl-241113-007':
    path = '/home/hedvigs/wslGit/snake_book/econ'
    site = 'silverFlex'

sys.path.append(path)
print(path)
from scripts.data_management import setup_data as gt


/home/hedvigs/PycharmProjects/homewrs/plab_workflow/workflow/


In [ ]:


def summarize_scores(trait=None, mtype=None, subset=None, model=None, gen=None, fold=None, verbose=0):
    inputs = [subset, gen, fold, mtype, trait, model]
    config_names = ['subsets', 'genotypes', 'folds', 'types', 'traits', 'c_models']

    config_dict = {}

    for item, config_name in zip(inputs, config_names):
        if item is not None:
            config_dict[config_name] = [item]
        else:
            config_dict[config_name] = gt.read_config(config_name)
    y_pred=[]
    missing = pd.DataFrame()
    k=0
    
    for trait in config_dict['traits']:
        for mtype in config_dict['types']:
            for subset in config_dict['subsets']:
                for model in config_dict['c_models']:
                    for gen in config_dict['genotypes']:
                        for fold in config_dict['folds']:
                            score_file= path + f'results/scores/{trait}/{mtype}/{subset}/{model}_{gen}_{fold}_pruned.csv'
                            try: 
                                scores = pd.read_csv(score_file, sep='\t')
                                scores['fold'] = fold
                                scores['gen'] = gen
                                scores['model'] = model
                                scores['subset'] = subset
                                scores['type'] = mtype
                                scores['trait'] = trait
                                y_pred.append(scores)
                            except FileNotFoundError:
                                k+=1
                                missing.loc[k, 'mtype'] = mtype
                                missing.loc[k, 'subset'] = subset
                                missing.loc[k, 'model'] = model
                                missing.loc[k, 'gen'] = gen
                                missing.loc[k, 'fold'] = fold
    if k == 0:
        print(f' All {mtype} files found!, {k} missing')
    elif verbose == 1:
        print(f'{mtype} files not found:', k)
    elif verbose == 2:
        print(f'{mtype} files not found:', k)
        subsetnames, subsetcounts =np.unique(missing['subset'], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f'  {sname}: {scount} missing')
            missubset = missing[missing['subset']==sname]
            modelnames, modelcounts =np.unique(missubset['model'], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f'    {mname}: {mcount} missing')
    elif verbose == 3:
        print('Total files not found:', k)
        for colname in missing.columns:
            names, counts = np.unique(missing[colname], return_counts=True)
            print(f'{colname}: ')
            for name, count in zip(names, counts):
                print(f'  {name}: {count} missing')
    elif verbose == 4:
        print(f'{mtype} files not found:', k)
        subsetnames, subsetcounts =np.unique(missing['subset'], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f'  {sname}: {scount} missing')
            missubset = missing[missing['subset']==sname]
            modelnames, modelcounts =np.unique(missubset['model'], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f'    {mname}: {mcount} missing')
                missmodel = missubset[missubset['model']==mname]
                gennames, gencounts =np.unique(missmodel['gen'], return_counts=True)
                for gname, gcount in zip(gennames, gencounts):
                    print(f'        {gname}: {gcount} missing')

    full_set = pd.concat(y_pred)
    return full_set






In [ ]:
if __name__ == '__main__':
    parser = argparse.ArgumentParser()

    parser.add_argument("-d", "--data")
    parser.add_argument("-o", "--out")
    parser.add_argument("-p", "--pheno")
    parser.add_argument("-l", "--list_top", nargs="*")
    parser.add_argument("-f", "--full")
    parser.add_argument("-w", "--wild", action=ParseKwargs)

    args = parser.parse_intermixed_args()
    wildcards = args.wild
    
    mtype = 'classic'
    trait = 'PTD'
    subset = None # wildcards["iSubset"]
    model = None
    gen = None  # wildcards["iGen"]



    sum_df = summarize_scores(trait=trait, mtype=mtype, subset=subset, model=model, gen=gen, fold=None, verbose=4)
    auc_df = sum_df.drop(columns=["trait", "type", "number"]) # ,"fold", "train_score", "f1", "fbeta", "perm_score", "plr", "nlr", "pval_score", "auc_pred"])
    sum_file= f'/mnt/work/workbench/hedvigs/snake_book/econ/out/tables/sum_file_2.csv'
    auc_df.to_csv(sum_file)
